In [1]:
import sys, os
sys.path.insert(0, os.path.expanduser('~/local/aispy'))

from aispy.psmap import load_psmap
import numpy as np

psmap = load_psmap('../output-files/PSGRID4D_Z0.h5')

# recover per-atom initial coords (one record per atom × port)
atom_idx = psmap['atom_indices']
n_atoms  = int(atom_idx.max()) + 1
nP       = int(np.bincount(atom_idx)[0])       # ports per atom
first    = np.searchsorted(atom_idx, np.arange(n_atoms))

x0  = psmap['initial_positions'][first, 0]
y0  = psmap['initial_positions'][first, 1]
vx0 = psmap['initial_velocities'][first, 0]
vy0 = psmap['initial_velocities'][first, 1]

xs, ys, vxs, vys = map(np.unique, [x0, y0, vx0, vy0])
nx, ny, nvx, nvy = len(xs), len(ys), len(vxs), len(vys)

# interfering ground-state port
inter_port = np.where(psmap['is_interfering'][:nP] & (psmap['states'][:nP] == 0))[0][0]

# coordinate-based indexing — correct regardless of ais++ iteration order
xi  = np.searchsorted(xs,  x0)
yi  = np.searchsorted(ys,  y0)
vxi = np.searchsorted(vxs, vx0)
vyi = np.searchsorted(vys, vy0)

dphi = psmap['phase_shifts'][inter_port::nP]
DPHI = np.empty((nx, ny, nvx, nvy))
DPHI[xi, yi, vxi, vyi] = dphi


In [2]:
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Interpolate to exact vx=vy=0 rather than nearest grid point.
# Required when the grid has even N (no exact zero); harmless when N is odd.
def slice_at_zero(arr4d, ax1, ax2, target1=0.0, target2=0.0):
    """Bilinear interpolation of a 4D array to a fixed value on axes 2 and 3."""
    # arr4d shape: (n0, n1, n2, n3); interpolate axes 2,3 → 2D slice over axes 0,1
    n0, n1, n2, n3 = arr4d.shape
    pts = np.array([[target1, target2]])
    out = np.empty((n0, n1))
    for i in range(n0):
        for j in range(n1):
            interp = RegularGridInterpolator((ax1, ax2), arr4d[i, j],
                                             method='linear', bounds_error=False,
                                             fill_value=None)
            out[i, j] = interp(pts)[0]
    return out

# spatial map at vx=vy=0
S_xy = slice_at_zero(DPHI, vxs, vys)
im0 = axes[0].pcolormesh(xs*1e3, ys*1e3, S_xy.T, cmap='RdBu_r', shading='auto')
axes[0].set(xlabel='x₀ [mm]', ylabel='y₀ [mm]', title='Δφ(x₀, y₀)  |  vx₀=vy₀=0')
fig.colorbar(im0, ax=axes[0], label='Δφ [rad]')

# velocity map at x=y=0
V_vxvy = slice_at_zero(DPHI.transpose(2, 3, 0, 1), xs, ys)
im1 = axes[1].pcolormesh(vxs*1e3, vys*1e3, V_vxvy.T, cmap='RdBu_r', shading='auto')
axes[1].set(xlabel='vx₀ [mm/s]', ylabel='vy₀ [mm/s]', title='Δφ(vx₀, vy₀)  |  x₀=y₀=0')
fig.colorbar(im1, ax=axes[1], label='Δφ [rad]')

plt.tight_layout()
plt.show()


In [3]:
import matplotlib.pyplot as plt

a0 = psmap['amp0'][inter_port::nP]
a1 = psmap['amp1'][inter_port::nP]
amp_mean = 0.5 * (a0 + a1)

AMP = np.empty((nx, ny, nvx, nvy))
AMP[xi, yi, vxi, vyi] = amp_mean

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ivx0 = np.searchsorted(vxs, 0.0);  ivy0 = np.searchsorted(vys, 0.0)
ix0  = np.searchsorted(xs,  0.0);  iy0  = np.searchsorted(ys,  0.0)

im0 = axes[0].pcolormesh(xs*1e3, ys*1e3, AMP[:, :, ivx0, ivy0].T,
                          cmap='viridis', shading='auto')
axes[0].set(xlabel='x₀ [mm]', ylabel='y₀ [mm]',
            title='mean amp(x₀, y₀)  |  vx₀=vy₀=0')
fig.colorbar(im0, ax=axes[0], label='amplitude')

im1 = axes[1].pcolormesh(vxs*1e3, vys*1e3, AMP[ix0, iy0, :, :].T,
                          cmap='viridis', shading='auto')
axes[1].set(xlabel='vx₀ [mm/s]', ylabel='vy₀ [mm/s]',
            title='mean amp(vx₀, vy₀)  |  x₀=y₀=0')
fig.colorbar(im1, ax=axes[1], label='amplitude')

plt.tight_layout()
plt.show()


In [4]:
len(psmap['path1'][-1])+229+229

1831

In [5]:
path0_splits

NameError: name 'path0_splits' is not defined

In [ ]:
psmap_1['path0'][4]

np.str_('0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010000000000000000000000000000000000000000000000000000000000000000000000000000